# Inferencing

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from ultralytics import YOLO

from PIL import Image

In [ ]:
# Local path to trained weights
YOLO_WEIGHTS_PATH = (r'C:\Users\Admin\CSC173-DeepCV-Reambonanza\csc173-deepcv-final-proj\yolo_final_weights\best.pt')
EFFICIENTNET_CLASSIFIER_PATH = (r'C:\Users\Admin\CSC173-DeepCV-Reambonanza\csc173-deepcv-final-proj\efficientnet_final_weights\EfficientNet_Final_Weights\efficientnet_b0_fine_tuned.pth')

In [ ]:
# Clarify if the model files exist
print(os.path.exists(YOLO_WEIGHTS_PATH))
print(os.path.exists(EFFICIENTNET_CLASSIFIER_PATH)) 

### Configuration

In [ ]:
# Class mapping (must match the order in EfficientNet training dataset)
CLASS_NAMES = ['dangerous', 'extremely_dangerous', 'safe', 'slightly_risky']
NUM_OF_CLASSES = 4 

In [ ]:
# Visual Colors for Bounding Boxes (BGR format)
RISK_COLORS = {
    'dangerous': (0, 165, 255),          # Orange (moderate risk)
    'extremely_dangerous': (0, 0, 255),  # Red (High risk)
    'safe': (0, 255, 0),                # Green (Low/No risk)
    'slightly_risky': (0, 255, 255)      # Yellow (Minor Risk)
}

In [ ]:
# Confidence threshold for YOLOv8 detection
YOLO_CONF_THRESHOLD = 0.10 # Use a slightly lower threshold to capture fuzzy boxes

In [ ]:
# Device setup
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

### EfficientNet Preprocessing

In [ ]:
# Mean and Std used for normalization
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

In [ ]:
# Preprocessing pipeline for inference (MUST match validation/test transforms)
preprocess = transforms.Compose([
    transforms.ToPILImage(),             # Convert OpenCV image to PIL image
    transforms.Resize((224, 224)),       # Resize to EfficientNet input size
    transforms.ToTensor(),               # Convert to Tensor
    transforms.Normalize(torch.Tensor(mean), torch.Tensor(std)) # Normalize
])

In [ ]:
# Model Loading Functions
def load_yolo_detector(path):
    """Loads the YOLOv8 detector model."""
    print("Loading YOLOv8 Detector...")
    return YOLO(path)

In [ ]:
def load_efficientnet_classifier(path, num_classes):
    """Reconstructs and loads the EfficientNet-B0 classifier."""
    print("Loading EfficientNet-B0 Classifier...")
    
    # 1. Load the pre-trained structure (without pre-trained weights)
    classifier = models.efficientnet_b0(weights=None) 
    
    # 2. Modify the classifier head to match your 4 classes
    num_ftrs = classifier.classifier[1].in_features
    classifier.classifier[1] = nn.Linear(num_ftrs, num_classes)
    
    # 3. Load the saved weights (state dictionary)
    state_dict = torch.load(path, map_location=DEVICE)
    classifier.load_state_dict(state_dict)
    
    # 4. Set to evaluation mode and move to device
    classifier = classifier.to(DEVICE)
    classifier.eval()
    
    return classifier

### Main Inference Loop and Visualization

In [ ]:
def run_two_stage_inference():
    # Load both models
    detector = load_yolo_detector(YOLO_WEIGHTS_PATH)
    classifier = load_efficientnet_classifier(EFFICIENTNET_CLASSIFIER_PATH, NUM_OF_CLASSES)

    # Open the webcam (0 is usually the default camera)
    cap = cv2.VideoCapture(0)

    # Set the resolution for faster processing (e.g., 1280x720 or 640x480)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280) 
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    print("Starting real-time inference. Press 'q' to exit.")
    
    # Disable gradient calculation for speed
    with torch.no_grad(): 
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            # Stage 1: Detection (YOLOv8)
            # Use 'r.boxes' format for cleaner iteration
            yolo_results = detector(frame, conf=YOLO_CONF_THRESHOLD, verbose=False)[0]

            for box in yolo_results.boxes:
                # Extract coordinates and confidence
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                conf = box.conf[0].cpu().item()
                
                # Stage 2: Cropping and Bounds Check
                # Ensure coordinates are within frame bounds before cropping
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(frame.shape[1], x2)
                y2 = min(frame.shape[0], y2)

                cropped_cluster = frame[y1:y2, x1:x2]
                
                # Skip if crop is empty (e.g., box is too small or invalid)
                if cropped_cluster.shape[0] < 20 or cropped_cluster.shape[1] < 20:
                    continue

                # Stage 3: Classification (EfficientNet-B0)
                
                # Preprocess and prepare for model input
                tensor_input = preprocess(cropped_cluster).unsqueeze(0).to(DEVICE)
                
                # Get classification output (logits)
                output = classifier(tensor_input)
                
                # Convert logits to probabilities and get max class
                probabilities = torch.softmax(output, dim=1)[0]
                max_prob, predicted_class_index = torch.max(probabilities, 0)
                
                # --- Stage 4: Decision Engine and Visualization ---
                final_class_name = CLASS_NAMES[predicted_class_index.item()]
                color = RISK_COLORS.get(final_class_name, (255, 255, 255)) 
                
                # Draw Bounding Box and Label with Confidence
                label = f'{final_class_name}: {max_prob.item():.2f}'

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                
                # Draw filled background for text label
                (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
                cv2.rectangle(frame, (x1, y1 - h - 10), (x1 + w + 10, y1), color, -1)
                cv2.putText(frame, label, (x1 + 5, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)


            # Display the resulting frame
            cv2.imshow("Two-Stage Wire Risk Detection", frame)

            # Exit on 'q' press
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    # Release resources
    cap.release()
    cv2.destroyAllWindows()
    print("Inference stopped.")

if __name__ == "__main__":
    run_two_stage_inference()